In [ ]:
import os
import json
import tempfile
from pathlib import Path
from typing import Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_recall_curve, roc_curve, confusion_matrix, classification_report
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import mlflow
from mlflow.models import infer_signature

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# 4. MLFlow

A continuacion, presentamos un flujo homologo al del notebook anterior pero adaptado para MLFlow.

## 4.1. MLflow — Tracking URI y experimento

In [ ]:
# MLFLOW_TRACKING_URI='http://ec2-44-203-136-23.compute-1.amazonaws.com:5000/'
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

EXPERIMENT_NAME = "PFG-TestDataScience-1-v2"
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)


## 4.2. Codigo funcional

### 4.2.1. Utils

In [ ]:
def _ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def plot_and_save_confusion_matrix(y_true, y_pred, outdir: Path, title: str) -> Path:
    cm = confusion_matrix(y_true, y_pred)
    fig = plt.figure()
    ax = fig.add_subplot(111)
    im = ax.imshow(cm, interpolation='nearest', aspect='auto')
    ax.figure.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center')
    fig.tight_layout()
    path = outdir / "confusion_matrix.png"
    fig.savefig(path)
    plt.close(fig)
    return path

def plot_and_save_pr_roc(y_true, y_score: Optional[np.ndarray], outdir: Path, title_prefix: str) -> Dict[str, Path]:
    paths = {}
    if y_score is None:
        return paths

    # Precision-Recall
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    fig_pr = plt.figure()
    plt.plot(recall, precision)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix} — Precision-Recall")
    pr_path = outdir / "precision_recall_curve.png"
    fig_pr.tight_layout()
    fig_pr.savefig(pr_path)
    plt.close(fig_pr)
    paths["pr"] = pr_path

    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fig_roc = plt.figure()
    plt.plot(fpr, tpr)
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"{title_prefix} — ROC")
    roc_path = outdir / "roc_curve.png"
    fig_roc.tight_layout()
    fig_roc.savefig(roc_path)
    plt.close(fig_roc)
    paths["roc"] = roc_path

    return paths

def get_xy(sufix):
    X_train = pd.read_excel(f'../data/processed/X_train_{sufix}.xlsx')
    X_test  = pd.read_excel(f'../data/processed/X_test_{sufix}.xlsx')
    y_train = pd.read_excel(f'../data/processed/y_train_{sufix}.xlsx')
    y_test  = pd.read_excel(f'../data/processed/y_test_{sufix}.xlsx')
    return X_train, X_test, y_train, y_test

def proba_pos(clf, X, pos_label=1):
    classes = clf.classes_
    idx = 1 if pos_label not in classes else int(np.where(classes == pos_label)[0][0])
    return clf.predict_proba(X)[:, idx]


### 4.2.2. Definición de modelos y *param grids*

In [ ]:
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
rf  = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
hgb = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
svc = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)
knn = KNeighborsClassifier()
xgb = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                    tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)
lgbm = LGBMClassifier(objective="binary", random_state=RANDOM_STATE, n_jobs=-1)
cat = CatBoostClassifier(loss_function="Logloss", random_state=RANDOM_STATE, verbose=False)

param_grid_logreg = {
    "C": [0.5, 1, 2],  
}
param_grid_rf = {
    "n_estimators": [600],
    "max_depth": [None, 20],
    "min_samples_leaf": [1, 10],
    "max_features": ["sqrt"],
    "class_weight": ["balanced"],
}
param_grid_hgb = {
    "learning_rate": [0.05, 0.1],
    "max_leaf_nodes": [31, 63],
    "min_samples_leaf": [20],
}
param_grid_svc = {
    "C": [1, 10, 100],
    "gamma": ["scale", 0.1],
}
param_grid_knn = {
    "n_neighbors": [15, 31],
    "weights": ["distance"],
    "p": [1, 2],
}
param_grid_xgb = {
    "n_estimators": [800],
    "learning_rate": [0.05, 0.1],
    "max_depth": [4, 6],
    "min_child_weight": [1, 5],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "reg_lambda": [1]
}

param_grid_lgbm = {
    "n_estimators": [800],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [31, 63],
    "min_child_samples": [20],
    "colsample_bytree": [0.8],
}
param_grid_cat = {
    "iterations": [800],
    "learning_rate": [0.05, 0.1],
    "depth": [4, 6],
    "l2_leaf_reg": [3],
}

modelos = {
    "LOGREG": (logreg, param_grid_logreg),
    "RF":     (rf,    param_grid_rf),
    #"HGB":    (hgb,   param_grid_hgb),
    #"SVC":    (svc,   param_grid_svc),
    #"KNN":    (knn,   param_grid_knn),
    #"XGB":    (xgb,   param_grid_xgb),
    #"LGBM":   (lgbm,  param_grid_lgbm),
    #"CAT":    (cat,   param_grid_cat),
}

### 4.2.3. Model selection

In [ ]:
def model_selection(modelos, X_train, y_train):
    # No cambiamos la firma ni los retornos
    y_train_1d = np.asarray(y_train).ravel()

    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE+1)

    summary_rows = []
    oof_by_model = {}
    per_fold_by_model = {}

    with mlflow.start_run(run_name="model_selection", nested=True):
        mlflow.set_tags({"phase": "model_selection"})

        for name, (pipeline, grid) in modelos.items():
            y_oof = np.full(len(y_train_1d), np.nan, dtype=float)
            folds_scores = []
            # Logueamos el espacio de búsqueda para trazabilidad
            mlflow.log_text(json.dumps(grid, indent=2, default=str), artifact_file=f"search_spaces/{name}.json")

            for fold, (tr_idx, va_idx) in enumerate(cv_outer.split(X_train, y_train_1d), start=1):
                X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
                y_tr, y_va = y_train_1d[tr_idx], y_train_1d[va_idx]

                with mlflow.start_run(run_name=f"{name}::fold{fold}", nested=True):
                    mlflow.set_tags({"model": name, "fold": fold, "cv": "outer"})

                    gs = GridSearchCV(
                        estimator=pipeline,
                        param_grid=grid,
                        scoring="average_precision",
                        cv=cv_inner,
                        n_jobs=-1,
                        refit=True,
                        verbose=0,
                        return_train_score=False
                    )
                    gs.fit(X_tr, y_tr)

                    # Log del mejor conjunto de hiperparámetros y score de CV
                    mlflow.log_params({f"best__{k}": (v if isinstance(v, (int,float,str,bool)) else str(v)) for k, v in gs.best_params_.items()})
                    mlflow.log_metric("cv_best_score", float(gs.best_score_))

                    best_est = gs.best_estimator_
                    # OOF sobre el fold de validación externa
                    if hasattr(best_est, "predict_proba"):
                        y_score = best_est.predict_proba(X_va)[:, 1]
                    elif hasattr(best_est, "decision_function"):
                        raw = best_est.decision_function(X_va)
                        y_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)
                    else:
                        y_score = None

                    if y_score is None:
                        ap = np.nan
                        roc = np.nan
                        f105 = f1_score(y_va, best_est.predict(X_va))
                    else:
                        ap = average_precision_score(y_va, y_score)
                        roc = roc_auc_score(y_va, y_score)
                        f105 = f1_score(y_va, (y_score >= 0.5).astype(int))

                    mlflow.log_metrics({
                        "oof_AP": float(ap) if not np.isnan(ap) else -1.0,
                        "oof_ROC": float(roc) if not np.isnan(roc) else -1.0,
                        "oof_F105": float(f105)
                    })

                    # Guardar OOF del fold
                    y_oof[va_idx] = y_score if y_score is not None else best_est.predict(X_va)

                    # Per-fold row
                    folds_scores.append({"fold": fold, "AP": float(ap) if not np.isnan(ap) else None,
                                         "ROC-AUC": float(roc) if not np.isnan(roc) else None,
                                         "F105": float(f105)})

            # Resumen por modelo
            fold_df = pd.DataFrame(folds_scores)
            per_fold_by_model[name] = fold_df.copy()

            summary_rows.append({
                "model": name,
                "AP_mean":  fold_df["AP"].mean(),
                "AP_std":   fold_df["AP"].std(),
                "ROC_mean": fold_df["ROC-AUC"].mean(),
                "ROC_std":  fold_df["ROC-AUC"].std(),
                "F105_mean": fold_df["F105"].mean(),
                "F105_std":  fold_df["F105"].std(),
            })

            # Artefactos por modelo
            tmpdir = Path(tempfile.mkdtemp(prefix=f"ms_{name}_"))
            _ensure_dir(tmpdir)
            fold_csv = tmpdir / f"{name}_per_fold_metrics.csv"
            fold_df.to_csv(fold_csv, index=False)
            mlflow.log_artifact(str(fold_csv), artifact_path=f"model_selection/{name}")

            # OOF guardado
            oof_by_model[name] = y_oof
            oof_df = pd.DataFrame({"y_true": y_train_1d, "y_oof": y_oof})
            oof_csv = tmpdir / f"{name}_oof.csv"
            oof_df.to_csv(oof_csv, index=False)
            mlflow.log_artifact(str(oof_csv), artifact_path=f"model_selection/{name}")

        summary_df = (pd.DataFrame(summary_rows)
                        .sort_values("AP_mean", ascending=False)
                        .reset_index(drop=True)
                        .round(4))
        best_model_name = summary_df.loc[0, "model"]
        # Log resumen
        tmpdir = Path(tempfile.mkdtemp(prefix="ms_summary_"))
        sum_csv = tmpdir / "summary_by_model.csv"
        summary_df.to_csv(sum_csv, index=False)
        mlflow.log_artifact(str(sum_csv), artifact_path="model_selection")

    return summary_df, oof_by_model, per_fold_by_model, best_model_name


### 4.2.4. Optimizacion threshold

In [ ]:
def get_best_threshold(y_train, best_model_name, oof_by_model):
    y_true = np.asarray(y_train).ravel()
    y_oof_best = oof_by_model[best_model_name]
    if np.isnan(y_oof_best).any():
        raise RuntimeError("OOF del mejor modelo contiene NaNs.")

    prec, rec, thr = precision_recall_curve(y_true, y_oof_best)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    best_idx = int(np.argmax(f1s))
    thr_opt = float(thr[best_idx])

    with mlflow.start_run(run_name="best_threshold", nested=True):
        mlflow.set_tags({"phase": "threshold_search", "best_model": best_model_name})
        mlflow.log_metric("f1_at_best_threshold", float(np.max(f1s)))
        mlflow.log_param("threshold_opt", thr_opt)

        # Guardar tabla PR + F1
        df_thr = pd.DataFrame({"precision": prec[:-1], "recall": rec[:-1], "threshold": thr, "f1": f1s})
        tmpdir = Path(tempfile.mkdtemp(prefix="thr_"))
        thr_csv = tmpdir / "pr_f1_thresholds.csv"
        df_thr.to_csv(thr_csv, index=False)
        mlflow.log_artifact(str(thr_csv), artifact_path="threshold")

    return thr_opt


### 4.2.5. Busqueda del mejor modelo

In [ ]:
def evaluate_best_model(modelos, X_train, y_train, best_model_name):
    # Entrena el mejor modelo con GridSearchCV sobre todo el train y devuelve el **best_estimator_**
    y_train_1d = np.asarray(y_train).ravel()
    pipeline, grid = modelos[best_model_name]
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    with mlflow.start_run(run_name=f"final_fit::{best_model_name}", nested=True):
        mlflow.set_tags({"phase": "final_fit", "model": best_model_name})
        mlflow.log_text(json.dumps(grid, indent=2, default=str), artifact_file="search_space.json" )

        gs = GridSearchCV(
            estimator=pipeline,
            param_grid=grid,
            scoring="average_precision",
            cv=cv_inner,
            n_jobs=-1,
            refit=True,
            verbose=0,
            return_train_score=False
        )
        gs.fit(X_train, y_train_1d)

        mlflow.log_params({f"best__{k}": (v if isinstance(v, (int,float,str,bool)) else str(v)) for k, v in gs.best_params_.items()})
        mlflow.log_metric("cv_best_score", float(gs.best_score_))

        best_est = gs.best_estimator_

        # Log del modelo
        try:
            X_example = X_train.iloc[:5]
            signature = infer_signature(X_example, best_est.predict(X_example))
        except Exception:
            signature = None

        mlflow.sklearn.log_model(best_est, name="model", signature=signature, registered_model_name=None)

    return best_est


### 4.2.6. Evaluacion en test

In [ ]:
def evaluate_on_test(X_test, y_test, thr_opt, final_model):
    y_tst = np.asarray(y_test).ravel()

    with mlflow.start_run(run_name="evaluate_on_test", nested=True):
        mlflow.set_tags({"phase": "test_evaluation"})
        # Scores
        if hasattr(final_model, "predict_proba"):
            y_score = final_model.predict_proba(X_test)[:, 1]
        elif hasattr(final_model, "decision_function"):
            raw = final_model.decision_function(X_test)
            y_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)
        else:
            y_score = None

        if y_score is None:
            ap = np.nan
            auc = np.nan
            y_pred_opt = final_model.predict(X_test)
        else:
            ap = average_precision_score(y_tst, y_score)
            auc = roc_auc_score(y_tst, y_score)
            y_pred_opt = (y_score >= thr_opt).astype(int)

        f1 = f1_score(y_tst, y_pred_opt)

        mlflow.log_metrics({
            "AP_test": float(ap) if not np.isnan(ap) else -1.0,
            "AUC_test": float(auc) if not np.isnan(auc) else -1.0,
            "F1opt_test": float(f1),
        })
        mlflow.log_param("threshold_opt", float(thr_opt))

        # Artefactos
        tmpdir = Path(tempfile.mkdtemp(prefix="test_"))
        cm_path = plot_and_save_confusion_matrix(y_tst, y_pred_opt, tmpdir, "Matriz de confusión (umbral óptimo)")
        mlflow.log_artifact(str(cm_path), artifact_path="plots")

        for _, p in plot_and_save_pr_roc(y_tst, y_score, tmpdir, "Test").items():
            mlflow.log_artifact(str(p), artifact_path="plots")

        # Informe de clasificación
        report = classification_report(y_tst, y_pred_opt, digits=4)
        report_path = tmpdir / "classification_report.txt"
        report_path.write_text(report, encoding="utf-8")
        mlflow.log_artifact(str(report_path), artifact_path="reports")

    return float(ap) if not np.isnan(ap) else None, float(auc) if not np.isnan(auc) else None, float(f1)


### 4.2.7. Pipeline

In [ ]:
def main(kinds):
    rows = []
    for kind in kinds:
        with mlflow.start_run(run_name=f"kind={kind}"):
            mlflow.set_tags({"kind": kind})

            X_train, X_test, y_train, y_test = get_xy(kind)
            summary_df, oof_by_model, per_fold_by_model, best_model_name = model_selection(modelos, X_train, y_train)
            thr_opt = get_best_threshold(y_train, best_model_name, oof_by_model)
            final_model = evaluate_best_model(modelos, X_train, y_train, best_model_name)
            ap, auc, f1 = evaluate_on_test(X_test, y_test, thr_opt, final_model)

            # Log resumen del kind
            best_cv = summary_df.loc[summary_df["model"] == best_model_name].iloc[0].to_dict()
            row = {
                "kind": kind,
                "best_model": best_model_name,
                "threshold": float(thr_opt),
                "AP_test": ap, "AUC_test": auc, "F1opt_test": f1,
                "AP_cv_mean":     float(best_cv.get("AP_mean", np.nan)),
                "AP_cv_std":      float(best_cv.get("AP_std", np.nan)),
                "ROC_cv_mean":    float(best_cv.get("ROC_mean", np.nan)),
                "ROC_cv_std":     float(best_cv.get("ROC_std", np.nan)),
                "F105_cv_mean":   float(best_cv.get("F105_mean", np.nan)),
                "F105_cv_std":    float(best_cv.get("F105_std", np.nan)),
            }
            rows.append(row)

            # Guardar artefactos de `summary_df` y del resumen final del kind
            tmpdir = Path(tempfile.mkdtemp(prefix=f"kind_{kind}_"))
            sum_csv = tmpdir / f"summary_{kind}.csv"
            summary_df.to_csv(sum_csv, index=False)
            mlflow.log_artifact(str(sum_csv), artifact_path="summary" )

            row_df = pd.DataFrame([row])
            row_csv = tmpdir / f"kind_{kind}_final_row.csv"
            row_df.to_csv(row_csv, index=False)
            mlflow.log_artifact(str(row_csv), artifact_path="summary")    

            print(f"---> Resultados para {kind}: AP={ap:.4f}, AUC={auc:.4f}, F1={f1:.4f}")

    final_df = pd.DataFrame(rows)
    return final_df


In [ ]:
# Descomenta para ejecutar si ya tienes los ficheros en ../data/processed/
results = main(['cc', 'cs', 'sc', 'ss'])
results